# Pong DQN Experiment Results

Use this notebook after running one or more training and benchmark jobs. It loads the JSON metrics produced by `src/test_pong_dqn.py` or `src/benchmark_pong_dqn.py`, compares reward statistics, and displays the best/worst GIFs for each experiment.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / 'results'
print('Project root:', PROJECT_ROOT)
print('Results directory:', RESULTS_DIR)

## 2. Load experiment metrics

In [ ]:
metric_files = sorted(RESULTS_DIR.glob('*_metrics.json'))
print(f'Found {len(metric_files)} metric file(s).')

experiments = []
for path in metric_files:
    with path.open('r', encoding='utf-8') as f:
        payload = json.load(f)
    payload['_metrics_path'] = path
    experiments.append(payload)

if not experiments:
    print('No metrics found yet. Run a benchmark first, for example:')
    print('python src/benchmark_pong_dqn.py --experiment baseline')

## 3. Compare summary statistics

In [ ]:
summary_rows = []
for payload in experiments:
    summary = payload.get('summary', {})
    summary_rows.append({
        'experiment': payload.get('experiment'),
        'episodes': summary.get('episodes'),
        'average_reward': summary.get('average_reward'),
        'std_reward': summary.get('std_reward'),
        'min_reward': summary.get('min_reward'),
        'max_reward': summary.get('max_reward'),
        'model_path': payload.get('model_path'),
        'metrics_path': str(payload.get('_metrics_path')),
    })

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values('average_reward', ascending=False)
summary_df

In [ ]:
if not summary_df.empty:
    ax = summary_df.plot(
        x='experiment',
        y='average_reward',
        yerr='std_reward',
        kind='bar',
        figsize=(10, 5),
        capsize=4,
        legend=False,
    )
    ax.set_title('Average reward by experiment')
    ax.set_xlabel('Experiment')
    ax.set_ylabel('Reward')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 4. Training time and convergence

Load the training metrics saved by `src/train_pong_dqn.py` and compare how quickly each experiment learned.

In [ ]:
train_metric_files = sorted(RESULTS_DIR.glob('*_train_metrics.json'))
print(f'Found {len(train_metric_files)} training metric file(s).')

train_rows = []
threshold_rows = []
for path in train_metric_files:
    with path.open('r', encoding='utf-8') as f:
        payload = json.load(f)

    run_name = payload.get('run_name')
    train_rows.append({
        'experiment': run_name,
        'elapsed_minutes': payload.get('elapsed_seconds', 0) / 60,
        'frames': payload.get('frames'),
        'episodes': payload.get('episodes'),
        'best_mean_reward': payload.get('best_mean_reward'),
        'final_mean_reward': payload.get('final_mean_reward'),
        'env_frameskip': payload.get('env_frameskip'),
        'wrapper_skip': payload.get('wrapper_skip'),
        'metrics_path': str(path),
    })

    for threshold, frame in payload.get('threshold_frames', {}).items():
        threshold_rows.append({
            'experiment': run_name,
            'threshold': float(threshold),
            'frame': frame,
        })

train_df = pd.DataFrame(train_rows)
if not train_df.empty:
    train_df = train_df.sort_values('best_mean_reward', ascending=False)
train_df

In [ ]:
if not train_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    train_df.plot(x='experiment', y='elapsed_minutes', kind='bar', ax=axes[0], legend=False)
    axes[0].set_title('Training time')
    axes[0].set_ylabel('Minutes')
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].grid(axis='y', alpha=0.3)

    train_df.plot(x='experiment', y='best_mean_reward', kind='bar', ax=axes[1], legend=False)
    axes[1].set_title('Best moving-average reward during training')
    axes[1].set_ylabel('Reward')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
threshold_df = pd.DataFrame(threshold_rows)
if not threshold_df.empty:
    threshold_pivot = threshold_df.pivot(index='experiment', columns='threshold', values='frame')
    display(threshold_pivot)
else:
    print('No threshold convergence data found yet.')

## 5. Episode reward distributions

In [ ]:
episode_rows = []
for payload in experiments:
    for episode in payload.get('episodes', []):
        episode_rows.append({
            'experiment': payload.get('experiment'),
            'episode': episode.get('episode'),
            'reward': episode.get('reward'),
            'steps': episode.get('steps'),
            'gif_path': episode.get('gif_path'),
        })

episodes_df = pd.DataFrame(episode_rows)
episodes_df.head()

In [ ]:
if not episodes_df.empty:
    plt.figure(figsize=(11, 5))
    for experiment, group in episodes_df.groupby('experiment'):
        plt.plot(group['episode'], group['reward'], marker='o', linewidth=1, label=experiment)
    plt.title('Episode rewards')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 6. Best and worst videos

In [ ]:
for payload in experiments:
    experiment = payload.get('experiment')
    summary = payload.get('summary', {})
    print(f'Experiment: {experiment}')
    print(f"Average reward: {summary.get('average_reward'):.2f} +/- {summary.get('std_reward'):.2f}")

    for label, key in [('Best', 'best_gif_path'), ('Worst', 'worst_gif_path')]:
        gif_value = summary.get(key)
        if not gif_value:
            continue
        gif_path = Path(gif_value)
        if not gif_path.is_absolute():
            gif_path = PROJECT_ROOT / gif_path
        print(f'{label}: {gif_path}')
        if gif_path.exists():
            display(Image(filename=str(gif_path)))
        else:
            print('GIF file not found.')
    print('-' * 80)